# 📊 Public Data ETL Pipeline — Guided Walkthrough

This notebook is a **narrated tour** of the data pipeline defined in the `src/`
folder. It does **not** re-implement anything — every step below *imports and
runs the same functions* that power `python -m src`. Think of it as the story
that goes with the machinery.

**What the pipeline does, in one sentence:** it pulls U.S. economic data from
the Federal Reserve (FRED), cleans it, runs data-quality checks, and only stores
data that passes — so bad data never reaches downstream analytics.

The stages, in order:

> **Extract** → **Transform** → **Validate** → **Load** → **Visualize**

We'll go through each one, showing the real output at every step.

## 0. Setup

Make the project's `src/` package importable and pull in the config.

In [ ]:
import sys
from pathlib import Path

# This notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.WARNING)  # keep output tidy in the notebook

from src import config

print("Series this pipeline pulls:")
for sid, cfg in config.SERIES.items():
    print(f"  {sid:7} | {cfg.frequency:9} | plausible range "
          f"[{cfg.min_value}, {cfg.max_value}] | {cfg.description}")

## 1. Extract — pull raw data from FRED

`extract.py` calls the FRED API for a single series and saves the **untouched**
response to `data/raw/`. Saving the raw copy first means we can always re-run
later stages without hitting the API again.

We'll use **UNRATE** (the unemployment rate) as our running example.

In [ ]:
from src.extract import fetch_series, save_raw_response

payload = fetch_series("UNRATE")          # live API call
raw_path = save_raw_response("UNRATE", payload)

print("Saved raw response to:", raw_path.name)
print("Total observations returned:", len(payload["observations"]))
print("\nFirst 3 raw records (exactly as FRED sends them):")
payload["observations"][:3]

Notice the raw data is all **text** — even the numbers are strings like `"3.5"`,
and missing values show up as `"."`. That's what the next stage fixes.

## 2. Transform — turn messy JSON into a clean table

`transform.py`'s `transform_observations()` is a **pure function**: raw data
goes in, a clean pandas DataFrame comes out, with no side effects. That purity
is what makes it easy to test.

It produces exactly three columns — **date, series_id, value** — with correct
types, and converts FRED's `"."` placeholder into a proper missing value (NaN).

In [ ]:
from src.transform import transform_observations

df = transform_observations(payload, "UNRATE")
print("Shape:", df.shape)
print("Column types:")
print(df.dtypes)
df.head()

In [ ]:
# Proof the "." placeholders became real missing values (NaN):
print("Missing values in 'value' column:", int(df["value"].isna().sum()))
df.tail()

## 3. Validate — the quality inspector (the heart of the project)

`validate.py` runs **five independent checks** and reports pass/fail with
details for each. This is the gate that keeps bad data out.

The five checks:
1. **schema** — right columns, right types
2. **null** — no blanks in the key columns (date / series_id)
3. **date_continuity** — no unexpected gaps for the series' frequency
4. **range** — values are inside a believable range
5. **duplicate** — no repeated (date, series_id) pairs

In [ ]:
from src.validate import DataQualityValidator

validator = DataQualityValidator(df, config.SERIES["UNRATE"])
results = validator.run_all()

for r in results:
    status = "PASS ✅" if r.passed else "FAIL ❌"
    print(f"{status}  {r.name:16} — {r.details}")

### What does a *failure* look like?

To prove the inspector actually catches problems, let's deliberately corrupt a
copy of the data — set one unemployment value to an impossible **999%** — and
re-run the range check.

In [ ]:
bad = df.copy()
bad.loc[bad.index[0], "value"] = 999.0   # impossible unemployment rate

bad_result = DataQualityValidator(bad, config.SERIES["UNRATE"]).check_range()
print("Passed?", bad_result.passed)
print("Details:", bad_result.details)

The check fails and explains exactly why. In the real pipeline, a series that
fails **is not loaded** into the database.

### The QA report

`generate_qa_report()` runs the checks for every series and writes a timestamped
markdown report to `qa_reports/`. Let's generate one for all three series and
display it right here.

In [ ]:
from src.validate import validate_series, generate_qa_report
from src.transform import transform_observations as _tx
from src.extract import fetch_series as _fetch
from IPython.display import Markdown, display

validations = []
for sid, cfg in config.SERIES.items():
    series_df = _tx(_fetch(sid), sid)
    validations.append(validate_series(series_df, cfg))

report_path = generate_qa_report(validations)
print("Report written to:", report_path.name, "\n")
display(Markdown(report_path.read_text()))

## 4. Load — store validated data in DuckDB

`load.py` saves the clean data into a local DuckDB database. It uses an
**upsert**: re-running updates existing rows instead of creating duplicates, so
the pipeline is safe to run on a schedule.

We'll load our UNRATE DataFrame, then query it back out to confirm it's there.

In [ ]:
from src.load import load_dataframe, get_connection

rows = load_dataframe(df)
print(f"Upserted {rows:,} UNRATE rows into the database.\n")

con = get_connection()
summary = con.execute('''
    SELECT series_id,
           COUNT(*)        AS rows,
           MIN(date)       AS first_date,
           MAX(date)       AS last_date
    FROM economic_data
    GROUP BY series_id
    ORDER BY series_id
''').df()
con.close()
summary

## 5. Visualize — see the data

Finally, we read the stored data back out of the database and plot it. This is
the same data `visualize.py` uses to build the HTML dashboard.

In [ ]:
import matplotlib.pyplot as plt

colors = {"UNRATE": "#7F77DD", "GDP": "#D85A30", "DGS10": "#1D9E75"}

con = get_connection()
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, (sid, cfg) in zip(axes, config.SERIES.items()):
    series_df = con.execute(
        "SELECT date, value FROM economic_data WHERE series_id = ? ORDER BY date",
        [sid],
    ).df()
    ax.plot(series_df["date"], series_df["value"], color=colors[sid], linewidth=1)
    ax.set_title(f"{sid} — {cfg.description}", loc="left", fontsize=11)
    ax.grid(alpha=0.25)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
con.close()
fig.tight_layout()
plt.show()

## 6. The whole pipeline in one call

Everything above runs automatically when you execute the pipeline. The
`run_pipeline()` function does extract → transform → validate → load for every
series (loading only the ones that pass) and returns a summary.

In [ ]:
from src.__main__ import run_pipeline

summary = run_pipeline()
summary

## Recap

- **Extract** got raw data from FRED and saved it verbatim.
- **Transform** turned it into a clean, correctly-typed table.
- **Validate** ran 5 quality checks and produced an auditable report — and it
  really does reject bad data.
- **Load** stored the good data in DuckDB without duplicates.
- **Visualize** turned the stored data into charts.

The key idea: **validation sits between ingestion and storage**, so only
trustworthy data ever reaches the analytics layer.

To run the pipeline yourself from the terminal:

```bash
python -m src            # run the full pipeline
python -m src.visualize  # build the HTML dashboard
pytest                   # run the 27 tests that prove the code works
```